# 02k — User Options

Procesa `data/data_raw/user_options.csv` para generar features de
preferencias de notificaciones y chat a nivel de usuario.

**CSV de entrada:** 87 MB, 1.164.298 filas, 5 columnas:
`_id`, `user_id`, `chat_notifications`, `arena_notifications`, `show_chat`.

**Hallazgo clave (sesión previa de validación):** la tabla tiene
**1 fila por usuario** (0% duplicados) y **cobertura del sample = 100%**.
El 99.30% de los jugadores está en TTT (todo activo, configuración por
defecto), por lo que la señal real está en el 0.70% restante (888 usuarios)
que apagaron alguna opción. Las 4 features generadas son cuasi-constantes
en magnitud pero direccionalmente útiles para el modelo de gustos
(especialmente `opt_arena_disabled`).

**Política point-in-time:** no aplica. La tabla no expone `created_at`
accesible y al haber 1 fila/usuario no hay decisiones temporales que tomar.
Se usa el snapshot completo.

**Outputs:**
- `data/data_qc/features_user_options.parquet` (126.253 × 5)
- `informes/fase1_cleaning/user_options/execution_report.md`
- HTML completo + sección enriquecida (logging dual)


In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)


In [2]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'user_options'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'user_options.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids.parquet'
PARQUET_FEATURES = DATA_QC / 'features_user_options.parquet'

def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()}")


PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_INPUT existe: True


In [3]:
# [SETUP] Carga sample_user_ids y REFERENCE_DATE
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

users_clean = pd.read_parquet(DATA_QC / 'users_clean.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
del users_clean
gc.collect()

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")

log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE: {REFERENCE_DATE}')


Usuarios en sample: 126,253
REFERENCE_DATE: 2026-04-04 18:23:31
[SETUP] 09:44:28 — sample_user_ids: 126,253 usuarios
[SETUP] 09:44:28 — REFERENCE_DATE: 2026-04-04 18:23:31


## 1. Carga y exploración del CSV

In [4]:
# [EXEC] Cargar user_options.csv completo
t0 = time.time()
df = pd.read_csv(CSV_INPUT)
n_before = len(df)
N_COLS_RAW = df.shape[1]
print(f"Cargado en {time.time()-t0:.1f}s")
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
log_step('EXEC', f'Carga CSV: shape={df.shape}, tiempo={time.time()-t0:.1f}s')


Cargado en 0.7s
Shape: (1164298, 5)
Memory: 89.7 MB
[EXEC] 09:44:29 — Carga CSV: shape=(1164298, 5), tiempo=0.7s


In [5]:
# [ANALYSIS] Exploración del CSV crudo
print("=== Tipos de datos ===")
info = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulls': df.isnull().sum(),
    'unique': [df[c].nunique() for c in df.columns],
    'sample': [df[c].iloc[0] for c in df.columns],
})
print(info)

# Guardar nulos crudos
df.isnull().sum().to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv', header=['nulls'])

# Cardinalidad: confirmar 1 fila/user
n_users_unique = df['user_id'].nunique()
ratio = len(df) / n_users_unique
print(f"\nCardinalidad: {len(df):,} filas / {n_users_unique:,} user_ids únicos = {ratio:.4f} filas/user")

# Distribuciones individuales
print("\n=== Distribuciones (CSV completo) ===")
for col in ['chat_notifications', 'arena_notifications', 'show_chat']:
    pct_true = df[col].mean() * 100
    print(f"  {col}: {pct_true:.2f}% True ({df[col].sum():,} / {len(df):,})")

ttt_global = (df['chat_notifications'] & df['arena_notifications'] & df['show_chat']).mean() * 100
print(f"\nTTT global (todo activo): {ttt_global:.2f}%")

log_step('ANALYSIS', f'Cardinalidad: {ratio:.4f} filas/user (1 fila/user → sin dedup)')
log_step('ANALYSIS', f'TTT global: {ttt_global:.2f}%')


=== Tipos de datos ===
                    dtype  nulls   unique                              sample
_id                   str      0  1164298  ObjectId(62c2fb637af2ca436778a522)
user_id               str      0  1164298            5eaa8f81fcdbdc3878446480
chat_notifications   bool      0        2                                True
arena_notifications  bool      0        2                                True
show_chat            bool      0        2                                True

Cardinalidad: 1,164,298 filas / 1,164,298 user_ids únicos = 1.0000 filas/user

=== Distribuciones (CSV completo) ===
  chat_notifications: 99.12% True (1,154,017 / 1,164,298)
  arena_notifications: 99.48% True (1,158,196 / 1,164,298)
  show_chat: 99.17% True (1,154,658 / 1,164,298)

TTT global (todo activo): 98.84%
[ANALYSIS] 09:44:29 — Cardinalidad: 1.0000 filas/user (1 fila/user → sin dedup)
[ANALYSIS] 09:44:29 — TTT global: 98.84%


## 2. Filtrado por sample_user_ids

In [6]:
# [EXEC] Normalizar user_id y filtrar por sample
df['user_id'] = df['user_id'].apply(clean_user_id)

n_no_nan = df['user_id'].notna().sum()
df = df[df['user_id'].isin(sample_user_ids)].copy()
n_in_sample = len(df)

print(f"CSV original: {n_before:,}")
print(f"En sample: {n_in_sample:,} ({n_in_sample/n_before*100:.2f}%)")
print(f"Cobertura del sample: {n_in_sample/N_SAMPLE*100:.2f}% ({n_in_sample:,}/{N_SAMPLE:,})")

log_step('EXEC', f'Filtrado: {n_before:,} → {n_in_sample:,} ({n_in_sample/n_before*100:.1f}%)')
log_step('EXEC', f'Cobertura sample: {n_in_sample/N_SAMPLE*100:.2f}%')


CSV original: 1,164,298
En sample: 126,253 (10.84%)
Cobertura del sample: 100.00% (126,253/126,253)
[EXEC] 09:44:29 — Filtrado: 1,164,298 → 126,253 (10.8%)
[EXEC] 09:44:29 — Cobertura sample: 100.00%


In [7]:
# [ANALYSIS] Estadísticas post-filtro y verificación de 8 combos
ttt_sample = (df['chat_notifications'] & df['arena_notifications'] & df['show_chat']).mean() * 100
print(f"=== Post-filtro ({n_in_sample:,} filas) ===")
print(f"TTT sample (todo activo): {ttt_sample:.2f}%")
print()
print("Distribución por feature individual (sample):")
for col in ['chat_notifications', 'arena_notifications', 'show_chat']:
    n_true = int(df[col].sum())
    n_false = n_in_sample - n_true
    print(f"  {col:25s} True={n_true:>7,} ({n_true/n_in_sample*100:.2f}%)  False={n_false:>5,} ({n_false/n_in_sample*100:.2f}%)")

print("\n=== Distribución cruzada (8 combos) ===")
combos_raw = df.groupby(['chat_notifications', 'arena_notifications', 'show_chat']).size().sort_values(ascending=False)
print(combos_raw)
print(f"\nTotal combos únicos: {len(combos_raw)}")

log_step('ANALYSIS', f'TTT sample: {ttt_sample:.2f}% (888 usuarios atípicos)')
log_step('ANALYSIS', f'Combos únicos en sample: {len(combos_raw)}')


=== Post-filtro (126,253 filas) ===
TTT sample (todo activo): 99.30%

Distribución por feature individual (sample):
  chat_notifications        True=125,592 (99.48%)  False=  661 (0.52%)
  arena_notifications       True=125,888 (99.71%)  False=  365 (0.29%)
  show_chat                 True=125,644 (99.52%)  False=  609 (0.48%)

=== Distribución cruzada (8 combos) ===
chat_notifications  arena_notifications  show_chat
True                True                 True         125365
False               True                 False           277
                    False                False           196
True                True                 False           130
False               True                 True            116
True                False                True             91
False               False                True             72
True                False                False             6
dtype: int64

Total combos únicos: 8
[ANALYSIS] 09:44:29 — TTT sample: 99.30% (888 usuarios

## 3. Feature engineering — 4 features con prefijo `opt_`

Decisión cerrada (justificada en sesión previa de validación):

| Feature | Tipo | Definición |
|---|---|---|
| `opt_chat_disabled` | bool | NOT chat_notifications (señal direccional positiva) |
| `opt_arena_disabled` | bool | NOT arena_notifications (clave para modelo de gustos) |
| `opt_show_chat_disabled` | bool | NOT show_chat (aversión social) |
| `opt_disabled_count` | int8 0-3 | Suma de las 3 anteriores (intensidad agregada) |

**Descartadas:** `opt_has_record` (constante), `opt_all_enabled` (= count==0),
`opt_all_disabled` (= count==3), `opt_has_any_disabled` (= count>0).


In [8]:
# [EXEC] Crear las 4 features con prefijo opt_ y reindex con sample
features = pd.DataFrame({
    'user_id': df['user_id'].values,
    'opt_chat_disabled': (~df['chat_notifications']).values,
    'opt_arena_disabled': (~df['arena_notifications']).values,
    'opt_show_chat_disabled': (~df['show_chat']).values,
})

features['opt_disabled_count'] = (
    features['opt_chat_disabled'].astype('int8') +
    features['opt_arena_disabled'].astype('int8') +
    features['opt_show_chat_disabled'].astype('int8')
).astype('int8')

# Tipos finales
for c in ['opt_chat_disabled', 'opt_arena_disabled', 'opt_show_chat_disabled']:
    features[c] = features[c].astype('bool')

# Reindex con sample (cobertura es 100% pero defendemos el invariante)
features = features.set_index('user_id').reindex(list(sample_user_ids))
n_missing_after_reindex = int(features['opt_chat_disabled'].isnull().sum())
assert n_missing_after_reindex == 0, \
    f"ERROR: {n_missing_after_reindex} usuarios del sample sin record (esperado: 0)"

features = features.reset_index().rename(columns={'index': 'user_id'})

# Tras reindex los dtypes pueden volverse object: re-castear
features['opt_chat_disabled'] = features['opt_chat_disabled'].astype('bool')
features['opt_arena_disabled'] = features['opt_arena_disabled'].astype('bool')
features['opt_show_chat_disabled'] = features['opt_show_chat_disabled'].astype('bool')
features['opt_disabled_count'] = features['opt_disabled_count'].astype('int8')

print(f"Shape features: {features.shape}")
print(f"Dtypes:")
print(features.dtypes)

log_step('EXEC', f'Features generadas: {features.shape[1]-1} (sin contar user_id)')
log_step('EXEC', f'Reindex OK: 0 usuarios faltantes (cobertura 100%)')


Shape features: (126253, 5)
Dtypes:
user_id                    str
opt_chat_disabled         bool
opt_arena_disabled        bool
opt_show_chat_disabled    bool
opt_disabled_count        int8
dtype: object
[EXEC] 09:44:29 — Features generadas: 4 (sin contar user_id)
[EXEC] 09:44:29 — Reindex OK: 0 usuarios faltantes (cobertura 100%)


## 4. Sanity checks — bloquean el guardado si fallan

In [9]:
# [ANALYSIS] Sanity checks (asserts bloqueantes)
checks_passed = []
checks_failed = []

# 1. Shape exacto
try:
    assert features.shape == (N_SAMPLE, 5), f"shape != ({N_SAMPLE}, 5): {features.shape}"
    checks_passed.append('Shape == (126253, 5)')
except AssertionError as e:
    checks_failed.append(str(e))

# 2. user_id único
try:
    assert features['user_id'].is_unique, "user_id no es único"
    checks_passed.append('user_id único')
except AssertionError as e:
    checks_failed.append(str(e))

# 3. Sin nulos
try:
    nulls_total = int(features.isnull().sum().sum())
    assert nulls_total == 0, f"hay {nulls_total} nulos"
    checks_passed.append('Sin nulos en features')
except AssertionError as e:
    checks_failed.append(str(e))

# 4. opt_disabled_count en [0, 3]
try:
    assert features['opt_disabled_count'].between(0, 3).all(), "disabled_count fuera de [0,3]"
    checks_passed.append('opt_disabled_count en [0, 3]')
except AssertionError as e:
    checks_failed.append(str(e))

# 5. Coherencia: count == 0 ↔ ningún disabled (debe coincidir con TTT del análisis previo: 125.365)
n_count_zero = int((features['opt_disabled_count'] == 0).sum())
EXPECTED_TTT = 125_365
try:
    assert n_count_zero == EXPECTED_TTT, \
        f"count==0 ({n_count_zero:,}) != TTT esperado ({EXPECTED_TTT:,}) del análisis previo"
    checks_passed.append(f'count==0 == TTT esperado ({EXPECTED_TTT:,})')
except AssertionError as e:
    checks_failed.append(str(e))

# 6. Coherencia algebraica: count == suma de los 3 booleanos
suma = (features['opt_chat_disabled'].astype('int8') +
        features['opt_arena_disabled'].astype('int8') +
        features['opt_show_chat_disabled'].astype('int8'))
try:
    assert (features['opt_disabled_count'] == suma).all(), "count != suma de los 3 booleanos"
    checks_passed.append('count == suma(3 booleanos)')
except AssertionError as e:
    checks_failed.append(str(e))

# 7. Distribución cruzada: 8 combos, suman N_SAMPLE
combos_final = features.groupby(['opt_chat_disabled', 'opt_arena_disabled', 'opt_show_chat_disabled']).size()
try:
    assert combos_final.sum() == N_SAMPLE, f"combos suman {combos_final.sum()} != {N_SAMPLE}"
    checks_passed.append(f'8 combos suman {N_SAMPLE:,}')
except AssertionError as e:
    checks_failed.append(str(e))

# 8. Tipos correctos
try:
    assert features['opt_chat_disabled'].dtype == bool
    assert features['opt_arena_disabled'].dtype == bool
    assert features['opt_show_chat_disabled'].dtype == bool
    assert str(features['opt_disabled_count'].dtype) == 'int8'
    checks_passed.append('Tipos correctos (bool×3 + int8)')
except AssertionError as e:
    checks_failed.append(f'Tipos incorrectos: {e}')

print("=== SANITY CHECKS ===")
for c in checks_passed:
    print(f"  {c}")
for c in checks_failed:
    print(f"  {c}")

assert len(checks_failed) == 0, f"BLOQUEO: {len(checks_failed)} sanity checks fallidos"
print(f"\n{len(checks_passed)}/{len(checks_passed)} checks pasaron")

log_step('ANALYSIS', f'Sanity checks: {len(checks_passed)} OK')


=== SANITY CHECKS ===
  Shape == (126253, 5)
  user_id único
  Sin nulos en features
  opt_disabled_count en [0, 3]
  count==0 == TTT esperado (125,365)
  count == suma(3 booleanos)
  8 combos suman 126,253
  Tipos correctos (bool×3 + int8)

8/8 checks pasaron
[ANALYSIS] 09:44:29 — Sanity checks: 8 OK


## 5. Validación de features

In [10]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("=== Dtypes finales ===")
for col in features.columns:
    dt = features[col].dtype
    nulls = int(features[col].isnull().sum())
    if features[col].dtype == bool:
        zeros = int((~features[col]).sum())
        nonzero = len(features) - zeros - nulls
        print(f"  {col:30s} dtype={str(dt):10s} nulls={nulls:>6,} zeros={zeros:>7,} nonzero={nonzero:>6,}")
    elif pd.api.types.is_numeric_dtype(features[col]):
        zeros = int((features[col] == 0).sum())
        nonzero = len(features) - zeros - nulls
        print(f"  {col:30s} dtype={str(dt):10s} nulls={nulls:>6,} zeros={zeros:>7,} nonzero={nonzero:>6,}")
    else:
        print(f"  {col:30s} dtype={str(dt):10s} nulls={nulls:>6,}")


=== Dtypes finales ===
  user_id                        dtype=str        nulls=     0
  opt_chat_disabled              dtype=bool       nulls=     0 zeros=125,592 nonzero=   661
  opt_arena_disabled             dtype=bool       nulls=     0 zeros=125,888 nonzero=   365
  opt_show_chat_disabled         dtype=bool       nulls=     0 zeros=125,644 nonzero=   609
  opt_disabled_count             dtype=int8       nulls=     0 zeros=125,365 nonzero=   888


In [11]:
# [ANALYSIS] Nulos en features finales
nulos_final = features.isnull().sum()
nulos_final.to_csv(REPORT_DIR / 'nulos_features_final.csv', header=['nulls'])
print("Nulos guardados en nulos_features_final.csv")
print(nulos_final)


Nulos guardados en nulos_features_final.csv
user_id                   0
opt_chat_disabled         0
opt_arena_disabled        0
opt_show_chat_disabled    0
opt_disabled_count        0
dtype: int64


In [12]:
# [ANALYSIS] Estadísticas descriptivas
numeric_cols = features.select_dtypes(include=['number', 'bool']).columns
desc = features[numeric_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T
desc.to_csv(REPORT_DIR / 'features_describe.csv')
print(desc.round(4).to_string())
log_step('ANALYSIS', 'features_describe.csv guardado')


                       count   mean     std  min  25%  50%  75%  90%  99%  max
opt_disabled_count  126253.0  0.013  0.1665  0.0  0.0  0.0  0.0  0.0  0.0  3.0
[ANALYSIS] 09:44:29 — features_describe.csv guardado


In [13]:
# [ANALYSIS] Histogramas de features
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 3 booleanas
for ax, col in zip(axes.flat[:3], ['opt_chat_disabled', 'opt_arena_disabled', 'opt_show_chat_disabled']):
    counts = features[col].value_counts().sort_index()
    n_false = int(counts.get(False, 0))
    n_true = int(counts.get(True, 0))
    ax.bar(['False', 'True'], [n_false, n_true], color=['#4caf50', '#e74c3c'])
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('# usuarios')
    for i, v in enumerate([n_false, n_true]):
        ax.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
    ax.set_yscale('log')

# Disabled count
ax = axes.flat[3]
counts = features['opt_disabled_count'].value_counts().sort_index()
ax.bar([str(i) for i in counts.index], counts.values, color='#3498db')
ax.set_title('opt_disabled_count', fontsize=10)
ax.set_ylabel('# usuarios')
ax.set_xlabel('# opciones desactivadas')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=9)
ax.set_yscale('log')

plt.suptitle('Distribución features opt_ (escala log)', fontsize=12)
plt.tight_layout()
out_png = REPORT_DIR / 'features_distributions.png'
plt.savefig(out_png, dpi=120, bbox_inches='tight')
plt.close()
print(f"Histogramas guardados: {out_png}")
log_step('ANALYSIS', 'features_distributions.png guardado')


Histogramas guardados: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_options/features_distributions.png
[ANALYSIS] 09:44:30 — features_distributions.png guardado


In [14]:
# [EXEC] Guardar parquet + sample_head + liberar memoria
t0 = time.time()
features.to_parquet(PARQUET_FEATURES, engine='pyarrow', compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
print(f"Parquet guardado en {time.time()-t0:.1f}s: {PARQUET_FEATURES} ({size_mb:.2f} MB)")

# Verificar lectura
features_read = pd.read_parquet(PARQUET_FEATURES)
assert features_read.shape == features.shape, f"Shape mismatch al leer: {features_read.shape} vs {features.shape}"
print(f"Parquet legible, shape={features_read.shape}")

# sample_head
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
print(f"sample_head.csv guardado")

# Liberar memoria del df crudo (features lo necesitamos para el report)
del df, features_read
gc.collect()

log_step('EXEC', f'Parquet guardado: ({features.shape[0]:,}, {features.shape[1]}), {size_mb:.2f} MB')
log_step('EXEC', 'sample_head.csv guardado')


Parquet guardado en 0.0s: /Users/jezquerro/Documents/tfg/data/data_qc/features_user_options.parquet (2.94 MB)
Parquet legible, shape=(126253, 5)
sample_head.csv guardado
[EXEC] 09:44:30 — Parquet guardado: (126,253, 5), 2.94 MB
[EXEC] 09:44:30 — sample_head.csv guardado


## 6. Informe de ejecución

In [15]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Métricas para el report
n_chat_disabled = int(features['opt_chat_disabled'].sum())
n_arena_disabled = int(features['opt_arena_disabled'].sum())
n_show_disabled = int(features['opt_show_chat_disabled'].sum())
count_dist = features['opt_disabled_count'].value_counts().sort_index()

lines = []
lines += [
    "# Execution Report — 02k_user_options",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02k_user_options.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/user_options.csv (87 MB, {n_before:,} filas, {N_COLS_RAW} cols)",
    f"**Parquet de salida:** data/data_qc/features_user_options.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Resumen",
    "",
    f"- **Tabla origen**: `user_options.csv` ({n_before:,} filas, 87 MB)",
    f"- **Sample size**: {N_SAMPLE:,} jugadores",
    f"- **Coverage del sample**: {n_in_sample/N_SAMPLE*100:.2f}%",
    f"- **Duplicados por user_id**: 0 (no se aplicó deduplicación)",
    f"- **Output**: `data/data_qc/features_user_options.parquet` ({features.shape[0]:,} filas × {features.shape[1]} columnas)",
    "",
    "---", "",
    "## Hipótesis validadas (sesión previa)",
    "",
    "| Hipótesis | Resultado | Estado |",
    "|---|---|---|",
    "| H1 — TTT (todo activo) domina la distribución | 99.30% sample / 98.84% global | Validada — la tabla tiene baja varianza |",
    "| H2 — Hay duplicados por user_id | 0% (1 fila/usuario exacta) | Refutada — no requiere dedup |",
    "| H3 — Cobertura del sample <100% | 100.00% | Refutada — cobertura total |",
    "",
    "---", "",
    "## Distribución del sample por combinación (chat / arena / show)",
    "",
    "| chat | arena | show | usuarios | % |",
    "|---|---|---|---|---|",
    "| | | | 125.365 | 99.30% |",
    "| | | | 277 | 0.22% |",
    "| | | | 196 | 0.16% |",
    "| | | | 130 | 0.10% |",
    "| | | | 116 | 0.09% |",
    "| | | | 91 | 0.07% |",
    "| | | | 72 | 0.06% |",
    "| | | | 6 | 0.005% |",
    "",
    "---", "",
    "## Constantes utilizadas",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    "",
    "---", "",
    "## Pasos ejecutados", "",
]
for s in steps_log:
    lines.append(f"- {s}")

lines += [
    "",
    "---", "",
    "## Filtrado aplicado",
    "",
    "| Paso | Filas | % CSV |",
    "|---|---|---|",
    f"| CSV original | {n_before:,} | 100.00% |",
    f"| Filtro sample_user_ids | {n_in_sample:,} | {n_in_sample/n_before*100:.2f}% |",
    "",
    "---", "",
    "## Features generadas (4 + user_id)",
    "",
    "| Feature | Tipo | Definición | % usuarios con valor positivo |",
    "|---|---|---|---|",
    f"| `opt_chat_disabled` | bool | NOT chat_notifications | {n_chat_disabled/N_SAMPLE*100:.2f}% ({n_chat_disabled:,}/{N_SAMPLE:,}) |",
    f"| `opt_arena_disabled` | bool | NOT arena_notifications | {n_arena_disabled/N_SAMPLE*100:.2f}% ({n_arena_disabled:,}/{N_SAMPLE:,}) |",
    f"| `opt_show_chat_disabled` | bool | NOT show_chat | {n_show_disabled/N_SAMPLE*100:.2f}% ({n_show_disabled:,}/{N_SAMPLE:,}) |",
    f"| `opt_disabled_count` | int8 (0-3) | Suma de las 3 anteriores | distribución abajo |",
    "",
    "---", "",
    "## Distribución de `opt_disabled_count`",
    "",
    "| Valor | Usuarios | % |",
    "|---|---|---|",
]
for v in [0, 1, 2, 3]:
    n = int(count_dist.get(v, 0))
    lines.append(f"| {v} | {n:,} | {n/N_SAMPLE*100:.2f}% |")

lines += [
    "",
    "---", "",
    "## Sanity checks aplicados (8)",
    "",
    "- [x] Shape == (126.253, 5)",
    "- [x] user_id único",
    "- [x] Sin nulos en features",
    "- [x] `opt_disabled_count` en [0, 3]",
    "- [x] `opt_disabled_count == 0` coincide con los 125.365 TTT del análisis previo",
    "- [x] `opt_disabled_count == suma(3 booleanos)` (coherencia algebraica)",
    "- [x] 8 combos suman 126.253",
    "- [x] Tipos: bool×3 + int8",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "1. **Reformulación direccional**: las 3 booleanas se invirtieron",
    "   (`_disabled` en lugar de `_notifications`/`show_chat`) para que",
    "   el valor 1 represente siempre 'el jugador apagó algo'. Esto",
    "   facilita correlaciones y interpretabilidad en el modelo de gustos.",
    "2. **Features descartadas**:",
    "   - `opt_has_record`: constante (cobertura 100%) → inútil",
    "   - `opt_all_enabled`: redundante con `opt_disabled_count == 0`",
    "   - `opt_all_disabled`: representable con `opt_disabled_count == 3`",
    "   - `opt_has_any_disabled`: representable con `opt_disabled_count > 0`",
    "3. **No deduplicación**: la tabla origen ya tiene 1 fila por user_id, no procede.",
    "4. **No filtro point-in-time**: la tabla no expone `created_at`/`updated_at`",
    "   accesibles directamente; el `_id` ObjectId no se usa porque no hay",
    "   duplicados que resolver temporalmente.",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- Cardinalidad 1 fila/user confirmada (sin necesidad de dedup ni agregación temporal)",
    "- Cobertura del sample del 100% (no hay usuarios fantasma en este CSV)",
    f"- Las 3 booleanas no son colineales (chat=False:{n_chat_disabled}, arena=False:{n_arena_disabled}, show=False:{n_show_disabled} → diferenciadas)",
    "- Reformulación direccional facilita lectura de correlaciones en EDA",
    "",
    "---", "",
    "## Limitaciones y caveats",
    "",
    "- **Baja varianza**: 99.30% de los jugadores está en TTT (default). Las 4",
    "  features tendrán impacto marginal en el modelo de churn, pero conservan",
    "  valor para el modelo de gustos como filtro de perfiles atípicos",
    "  (jugadores que activamente desactivan opciones).",
    "- **Sesgo del default**: no se puede distinguir entre un jugador que",
    "  conscientemente eligió 'todo activo' y uno que nunca abrió los ajustes.",
    "  Ambos aparecen como TTT.",
    "- **Ausencia de temporalidad**: la tabla no expone `created_at`/`updated_at`",
    "  accesibles; el `_id` ObjectId no se usa porque no hay duplicados.",
    "",
    "---", "",
    "## Archivos generados",
    "",
    "| Archivo | Descripción |",
    "|---|---|",
    "| features_user_options.parquet (5 cols) | Parquet final |",
    "| nulos_por_columna_raw.csv | Nulos CSV crudo |",
    "| nulos_features_final.csv | Nulos features finales |",
    "| features_describe.csv | Estadísticas descriptivas |",
    "| features_distributions.png | Histogramas features |",
    "| sample_head.csv | 20 primeras filas del parquet |",
    "| 02k_user_options_run.html | Notebook ejecutado completo (logging dual) |",
    "| execution_report.md | Este informe |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    f"- REFERENCE_DATE = {REFERENCE_DATE} (no usado para filtrar en 02k)",
    "- Las 4 features son cuasi-constantes (>99% en valor 0/False); evaluarlas en EDA con histograma log-scale.",
    "- `opt_arena_disabled` es la candidata más informativa para el modelo de gustos.",
    "- En 02z_build_master_table no requiere transformación adicional (ya son numéricas/booleanas).",
    "",
    "---", "",
    "## TODOs para fases siguientes",
    "",
    "- **Fase 2 (EDA)**: validar correlación entre `opt_arena_disabled=True` y baja actividad",
    "  en `arena_log` — debería existir si la señal es coherente.",
    "- **Fase 3 (modelado)**: evaluar si `opt_disabled_count` aporta información incremental",
    "  sobre los 3 booleanos individuales (puede ser redundante en árboles; valioso en lineales por colinealidad explícita).",
    "- **Modelo de gustos**: `opt_arena_disabled` es candidato a feature de alto valor",
    "  para segmentación competitivo/no-competitivo.",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"user_options.csv ({n_before:,} filas × {N_COLS_RAW} cols)",
    "    │",
    f"    └─ Filtrar por sample_user_ids   (-{n_before - n_in_sample:,})",
    "",
    f"Filtrado ({n_in_sample:,} filas, 1 fila/user)",
    "    │",
    "    ├─ opt_chat_disabled = ~chat_notifications",
    "    ├─ opt_arena_disabled = ~arena_notifications",
    "    ├─ opt_show_chat_disabled = ~show_chat",
    "    ├─ opt_disabled_count = suma(3 booleanos)",
    "    └─ Reindex con sample_user_ids (no-op: cobertura 100%)",
    "",
    f"features_user_options.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero (genera sample_user_ids)",
    "2. Ejecutar 02k_user_options.ipynb",
    "3. Verificar: features_user_options.parquet == 126.253 filas × 5 cols",
    "",
    "---", "",
    "## Referencias",
    "",
    "- PLANTILLA_NOTEBOOK_02.md — estructura estándar de notebook Fase 1.",
    "- 02e_user_daily_rewards.ipynb — referencia para tabla con 1 fila/usuario.",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')


Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_options/execution_report.md
[REPORT] 09:44:30 — execution_report.md generado


In [16]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02k_user_options")
print("=" * 70)
print(f"  Tiempo total              : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV original        : {n_before:,}")
print(f"  Filas en sample           : {n_in_sample:,} ({n_in_sample/n_before*100:.1f}%)")
print(f"  Cobertura del sample      : {n_in_sample/N_SAMPLE*100:.2f}%")
print(f"  Filas features final      : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features         : {features.shape[1]}")
print()
print(f"  opt_chat_disabled = True  : {int(features['opt_chat_disabled'].sum()):,} ({features['opt_chat_disabled'].mean()*100:.2f}%)")
print(f"  opt_arena_disabled = True : {int(features['opt_arena_disabled'].sum()):,} ({features['opt_arena_disabled'].mean()*100:.2f}%)")
print(f"  opt_show_chat_disabled=T  : {int(features['opt_show_chat_disabled'].sum()):,} ({features['opt_show_chat_disabled'].mean()*100:.2f}%)")
print()
print(f"  opt_disabled_count distribución:")
for v in [0, 1, 2, 3]:
    n = int((features['opt_disabled_count'] == v).sum())
    print(f"    {v}: {n:>8,} ({n/N_SAMPLE*100:.2f}%)")
print()
print("Archivos generados:")
print(f"  features_user_options.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.2f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)


RESUMEN FINAL — Notebook 02k_user_options
  Tiempo total              : 0m 2s
  Filas CSV original        : 1,164,298
  Filas en sample           : 126,253 (10.8%)
  Cobertura del sample      : 100.00%
  Filas features final      : 126,253 (== 126,253 sample)
  Columnas features         : 5

  opt_chat_disabled = True  : 661 (0.52%)
  opt_arena_disabled = True : 365 (0.29%)
  opt_show_chat_disabled=T  : 609 (0.48%)

  opt_disabled_count distribución:
    0:  125,365 (99.30%)
    1:      337 (0.27%)
    2:      355 (0.28%)
    3:      196 (0.16%)

Archivos generados:
  features_user_options.parquet (2.94 MB)
  execution_report.md
  Gráficos y CSVs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_options


In [17]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02k_user_options.ipynb'
html_path = REPORT_DIR / '02k_user_options_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(n_before, N_COLS_RAW),
    filter_steps=[
        ('CSV original', n_before),
        ('En sample', n_in_sample),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")


HTML generado: 02k_user_options_run.html (0.4 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_options/execution_report.md
